<a href="https://colab.research.google.com/github/Sophieli0113/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
removed = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True).copy()
log('duplicates', 'dropped exact duplicate rows', removed)
print('rows:', len(df))

[duplicates] dropped exact duplicate rows (15 row(s))
rows: 300


### TODO 2 — clean `price` -> float

In [4]:
# TODO
had_dollar = df['price'].astype(str).str.contains('$', regex=False).sum()
df['price'] = (df['price'].astype(str)
               .str.replace('$', '', regex=False)
               .str.strip()
               .astype(float))
assert df['price'].dtype == float
log('price', f'stripped $ from {had_dollar} rows, cast text to float', len(df))
print(sorted(df['price'].unique().tolist()))

[price] stripped $ from 124 rows, cast text to float (300 row(s))
[6.0, 7.5, 12.0, 24.0]


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
missing = df['qty'].isna().sum()
negative = (df['qty'] < 0).sum()
df = df[df['qty'].notna() & (df['qty'] >= 0)].reset_index(drop=True).copy()
log('qty_missing', 'dropped rows with no quantity', missing)
log('qty_negative', 'dropped rows with negative quantity', negative)
print('rows now:', len(df))

[qty_missing] dropped rows with no quantity (12 row(s))
[qty_negative] dropped rows with negative quantity (13 row(s))
rows now: 275


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger':  'cheeseburger',
    'cheese burger': 'cheeseburger',
    'cheeseburger':  'cheeseburger',
    'Foam Finger':   'foam finger',
    'foam finger':   'foam finger',
    'Rain Poncho':   'rain poncho',
    'rain poncho':   'rain poncho',
}
n_before = df['item'].nunique()
original = df['item'].copy()
df['item'] = df['item'].map(ITEM_MAP)
assert df['item'].notna().all(), 'an item variant is missing from ITEM_MAP'
changed = (df['item'] != original).sum()
log('item', f'canonicalized item names: {n_before} variants -> {df["item"].nunique()}', changed)
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] canonicalized item names: 6 variants -> 3 (193 row(s))
item
foam finger     97
rain poncho     91
cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())
n_before = df['category'].nunique()
original = df['category'].copy()

# fix different spellings
CATEGORY_MAP = {
    'Food': 'food', 'food': 'food',
    'RainGear': 'rain gear', 'rain-gear': 'rain gear', 'rain gear': 'rain gear',
    'Merch': 'merch', 'merch': 'merch',
    'Apparel': 'apparel', 'apparel': 'apparel',
}
df['category'] = df['category'].map(CATEGORY_MAP)
assert df['category'].notna().all(), 'a category variant is missing from CATEGORY_MAP'
changed = (df['category'] != original).sum()
log('category', f'normalized spelling: {n_before} variants -> {df["category"].nunique()}', changed)

# apparel is merchandise
merged = (df['category'] == 'apparel').sum()
df['category'] = df['category'].replace({'apparel': 'merch'})
log('category_merge', 'merged apparel into merch (both non-food goods)', merged)
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized spelling: 6 variants -> 4 (231 row(s))
[category_merge] merged apparel into merch (both non-food goods) (43 row(s))
category
food         95
merch        94
rain gear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item']) == {'cheeseburger', 'foam finger', 'rain poncho'}, 'unexpected item name'
assert set(df['category']) == {'food', 'merch', 'rain gear'}, 'unexpected category'
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = df['qty'] * df['price']

by_category = (df.groupby('category')['revenue']
                 .sum()
                 .sort_values(ascending=False)
                 .round(2))
print(by_category.map('{:.2f}'.format))
print(f"total: {df['revenue'].sum():.2f}")

category
food         1656.00
merch        1572.00
rain gear    1512.00
Name: revenue, dtype: object
total: 4740.00


**What I would tell the vendor: I would tell a vendor to stock more food because it generates higher revenue.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,"stripped $ from 124 rows, cast text to float",300
2,qty_missing,dropped rows with no quantity,12
3,qty_negative,dropped rows with negative quantity,13
4,item,canonicalized item names: 6 variants -> 3,193
5,category,normalized spelling: 6 variants -> 4,231
6,category_merge,merged apparel into merch (both non-food goods),43


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

(a) Dropping exact duplicate rows changed the revenue total the most. Revenue before: \$4,852.50. Revenue after: \$4,594.50 - change is -\$258.00.

(b) I dropped the 13 negative quantity rows. They could be kept as refunds instead, which would have lowered revenue from \$4,740.00 to \$4,594.50. I dropped them because nothing in the data shows they are real refunds, so they looked like entry errors to me.